# Khmer Text-to-Speech (TTS) Inference Demo
This notebook demonstrates how to load the FastSpeechLite + HiFiGAN-tiny model to synthesize Khmer text into audio.
You can run this interactively!

In [1]:
import os
import torch
import soundfile as sf
import numpy as np
import IPython.display as ipd
import torchaudio.transforms as T

from model.acoustic_model import FastSpeechLite
from model.vocoder import Generator
from data.khmer_tokenizer import KhmerTokenizer

# Use CPU or GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Initialize Components (Tokenizer, Acoustic Model, Vocoder)

In [2]:
# 1. Load Tokenizer
vocab_path = "web/models/vocab.json"
if not os.path.exists(vocab_path):
    vocab_path = "km_kh_male/processed/vocab.json"

tokenizer = KhmerTokenizer(vocab_path=vocab_path)
vocab_size = len(tokenizer.vocab)
print(f"Vocabulary size: {vocab_size}")

# 2. Load Acoustic Model (FastSpeechLite)
acoustic_ckpt = "checkpoints/acoustic/best_acoustic.pt"
acoustic = FastSpeechLite(vocab_size=vocab_size).to(device)

if os.path.exists(acoustic_ckpt):
    ckpt = torch.load(acoustic_ckpt, map_location=device)
    acoustic.load_state_dict(ckpt["model"], strict=False)
    print("Acoustic model loaded successfully!")
else:
    print("Acoustic checkpoint not found! Proceeding with untrained weights.")
acoustic.eval()

# 3. Load Vocoder (HiFiGAN or Griffin-Lim)
vocoder = Generator().to(device)
vocoder_ckpt = "checkpoints/vocoder/best_vocoder.pt"

if os.path.exists(vocoder_ckpt):
    ckpt = torch.load(vocoder_ckpt, map_location=device)
    vocoder.load_state_dict(ckpt["gen"], strict=False)
    vocoder.remove_weight_norm()
    print("Vocoder loaded successfully!")
else:
    print("Vocoder checkpoint not found! Proceeding with untrained weights.")
vocoder.eval()

Vocabulary size: 77
Acoustic model loaded successfully!
Vocoder loaded successfully!


C:\Users\ASUS TUF\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Generator(
  (conv_pre): Conv1d(80, 256, kernel_size=(7,), stride=(1,), padding=(3,))
  (ups): ModuleList(
    (0): ConvTranspose1d(256, 128, kernel_size=(16,), stride=(8,), padding=(4,))
    (1): ConvTranspose1d(128, 64, kernel_size=(16,), stride=(8,), padding=(4,))
    (2): ConvTranspose1d(64, 32, kernel_size=(4,), stride=(2,), padding=(1,))
    (3): ConvTranspose1d(32, 16, kernel_size=(4,), stride=(2,), padding=(1,))
  )
  (resblocks): ModuleList(
    (0): ResBlock(
      (convs): ModuleList(
        (0): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(3,), dilation=(3,))
        (2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(5,), dilation=(5,))
      )
    )
    (1): ResBlock(
      (convs): ModuleList(
        (0): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(3,))
        (1): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
        (2): Conv1d

## Define Synthesis Function

In [3]:
def synthesize(text, speed=1.0):
    text = text.strip()
    if not text:
        return None
        
    token_ids = tokenizer.text_to_ids(text)
    phon_tensor = torch.tensor([token_ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        pace = 1.0 / max(speed, 0.2)
        enc_out = acoustic.encoder(phon_tensor)
        log_dur_pred = acoustic.duration_predictor(enc_out)
        durations = torch.clamp(torch.round((torch.exp(log_dur_pred) - 1) * pace), min=1).long()
        
        expanded, out_lens = acoustic.length_regulator(enc_out, durations)
        mel_pred = acoustic.decoder(expanded)
        mel_len = max(1, int(out_lens[0].item()))
        
        mel_for_vocoder = mel_pred[:, :mel_len, :].transpose(1, 2)
        wav_pred = vocoder(mel_for_vocoder)
        waveform = wav_pred.squeeze().cpu().numpy()
        
    if len(waveform) > 0:
        waveform = waveform - np.mean(waveform)
        peak = np.max(np.abs(waveform))
        if peak > 1e-4:
            waveform = (waveform / peak) * 0.95
            
    return waveform.astype(np.float32)

## Generate Speech
Change the `text` variable to try different Khmer inputs.

In [4]:
text = "ស្ពាន កំពង់ ចម្លង អ្នកលឿង"
speed = 1.0

print(f"Synthesizing: {text}")
waveform = synthesize(text, speed=speed)

# Display the audio player
ipd.display(ipd.Audio(waveform, rate=22050))

Synthesizing: ស្ពាន កំពង់ ចម្លង អ្នកលឿង
